# RETFound Encoding — Google Colab
Encodes EyePACS and Messidor images with frozen RETFound (ViT-Large).
Run on GPU runtime: Runtime → Change runtime type → T4 GPU.
Expected time: ~15 min for EyePACS (35K images), ~1 min for Messidor.

In [ ]:
# Install dependencies
!pip install timm wandb -q

In [ ]:
# Mount Google Drive — save embeddings here so they persist
from google.colab import drive
drive.mount('/content/drive')

import os
OUT_DIR = '/content/drive/MyDrive/synapse/embeddings'
os.makedirs(OUT_DIR, exist_ok=True)
print('Output dir:', OUT_DIR)

In [ ]:
# Upload data zips from your laptop
# Run this cell to upload files interactively
from google.colab import files
print('Upload your zip files one at a time:')
print('  - trainLabels.csv (from data/raw/eyepacs/)')
print('  - eyepacs_images.zip (zip of data/raw/eyepacs/train/images/train/)')
print('  - messidor_images.zip (zip of data/raw/messidor/images/IMAGES/)')
print('  - messidor-2.csv (from data/raw/messidor/)')
print()
print('OR use the Kaggle API cell below instead for EyePACS.')

In [ ]:
# OPTION A: Download EyePACS via Kaggle API (recommended — faster than uploading)
# Upload your kaggle.json first
from google.colab import files
print('Upload your kaggle.json file:')
uploaded = files.upload()  # upload kaggle.json

!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

# Download only train labels + first chunk to test (remove -f flag for full download)
!pip install kaggle -q
!kaggle competitions download -c diabetic-retinopathy-detection -f trainLabels.csv.zip -p /content/eyepacs/
!kaggle competitions download -c diabetic-retinopathy-detection -f train.zip.001 -p /content/eyepacs/
!kaggle competitions download -c diabetic-retinopathy-detection -f train.zip.002 -p /content/eyepacs/
!kaggle competitions download -c diabetic-retinopathy-detection -f train.zip.003 -p /content/eyepacs/
!kaggle competitions download -c diabetic-retinopathy-detection -f train.zip.004 -p /content/eyepacs/
!kaggle competitions download -c diabetic-retinopathy-detection -f train.zip.005 -p /content/eyepacs/

In [ ]:
# Extract EyePACS
!mkdir -p /content/eyepacs/images
!cd /content/eyepacs && unzip -q trainLabels.csv.zip
# Extract split archive parts
import subprocess
for i in range(1, 6):
    f = f'/content/eyepacs/train.zip.00{i}'
    if os.path.exists(f):
        print(f'Extracting {f}...')
        subprocess.run(['unzip', '-q', '-o', f, '-d', '/content/eyepacs/images/'], check=False)
import glob
imgs = glob.glob('/content/eyepacs/images/**/*.jpeg', recursive=True)
print(f'Total images: {len(imgs)}')

In [ ]:
# Load RETFound
import torch
import timm
import numpy as np
from pathlib import Path
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from tqdm import tqdm
import pandas as pd

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
if DEVICE == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

model = timm.create_model('hf_hub:bitfount/RETFound_MAE', pretrained=True, num_classes=0)
model.eval().to(DEVICE)
for p in model.parameters(): p.requires_grad = False

# Probe embedding dim
with torch.no_grad():
    dim = model(torch.zeros(1,3,224,224).to(DEVICE)).shape[-1]
print(f'Embedding dim: {dim}')

In [ ]:
TRANSFORM = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])

class EyePACSDataset(Dataset):
    def __init__(self, img_dir, csv_path):
        df = pd.read_csv(csv_path)
        df['path'] = df['image'].apply(lambda x: Path(img_dir)/f'{x}.jpeg')
        self.df = df[df['path'].apply(lambda p: p.exists())].reset_index(drop=True)
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        row = self.df.iloc[i]
        return TRANSFORM(Image.open(row['path']).convert('RGB')), int(row['level'])

class FolderDataset(Dataset):
    def __init__(self, folder):
        self.paths = sorted(Path(folder).rglob('*'))
        self.paths = [p for p in self.paths if p.suffix.lower() in ['.png','.jpg','.jpeg']]
    def __len__(self): return len(self.paths)
    def __getitem__(self, i):
        return TRANSFORM(Image.open(self.paths[i]).convert('RGB')), 0

@torch.no_grad()
def encode(dataset, batch_size=64):
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=2)
    embs = []
    for imgs, _ in tqdm(loader):
        embs.append(model(imgs.to(DEVICE)).cpu().numpy())
    return np.concatenate(embs, axis=0)

In [ ]:
# Encode EyePACS
eyepacs = EyePACSDataset('/content/eyepacs/images', '/content/eyepacs/trainLabels.csv')
print(f'EyePACS: {len(eyepacs)} images')
embs_eyepacs = encode(eyepacs, batch_size=64)
np.save(f'{OUT_DIR}/eyepacs_retfound.npy', embs_eyepacs)
print(f'Saved: {embs_eyepacs.shape}')

In [ ]:
# Upload and encode Messidor
# Upload messidor images zip when prompted
uploaded = files.upload()
!mkdir -p /content/messidor
!unzip -q /content/*.zip -d /content/messidor/ 2>/dev/null || true

messidor = FolderDataset('/content/messidor')
print(f'Messidor: {len(messidor)} images')
embs_messidor = encode(messidor, batch_size=64)
np.save(f'{OUT_DIR}/messidor_retfound.npy', embs_messidor)
print(f'Saved: {embs_messidor.shape}')

In [ ]:
# Done — embeddings saved to Google Drive at:
print('Embeddings saved to Google Drive:')
import os
for f in os.listdir(OUT_DIR):
    size = os.path.getsize(f'{OUT_DIR}/{f}')
    print(f'  {f}: {round(size/1e6, 1)} MB')
print()
print('Download them from Google Drive and place in:')
print('  data/processed/embeddings/eyepacs_retfound.npy')
print('  data/processed/embeddings/messidor_retfound.npy')